In [0]:
# ===================================================
# BLOCK 1 — IMPORTS AND PARAMETERS (PYTHON)
# ===================================================

"""
Define the source and controlled-arrival locations used to reproduce two
bounded file-arrival waves without modifying the immutable generated dataset.
"""

from pathlib import PurePosixPath

dbutils.widgets.text("batch_start", "1", "First source-file sequence")
dbutils.widgets.text("batch_end", "5", "Last source-file sequence")
dbutils.widgets.dropdown("reset_demo", "false", ["false", "true"], "Reset demo")

SOURCE_DIRECTORY = (
    "/Volumes/semiconplus_portfolio/landing/external_source/streaming_events"
)
ARRIVAL_DIRECTORY = (
    "/Volumes/semiconplus_portfolio/landing/external_source/streaming_demo/input"
)

BATCH_START = int(dbutils.widgets.get("batch_start"))
BATCH_END = int(dbutils.widgets.get("batch_end"))
RESET_DEMO = dbutils.widgets.get("reset_demo").lower() == "true"

assert 1 <= BATCH_START <= BATCH_END <= 60, (
    "The bounded demonstration supports source-file sequences 1 through 60."
)


In [0]:
# ===================================================
# TEMPORARY OVERRIDE — SECOND ARRIVAL WAVE (PYTHON)
# ===================================================

# BATCH_START = 6
# BATCH_END = 10
# RESET_DEMO = False

# print(
#     f"Staging source files {BATCH_START} through {BATCH_END}; "
#     f"reset_demo={RESET_DEMO}"
# )

In [0]:
# ===================================================
# BLOCK 2 — OPTIONAL DEMONSTRATION RESET (PYTHON)
# ===================================================

"""
Remove only the controlled streaming-arrival directory when an intentional
full demonstration reset is requested. Pipeline state must also be reset from
the pipeline UI before files with previously processed names are replayed.
"""

if RESET_DEMO:
    dbutils.fs.rm(ARRIVAL_DIRECTORY, recurse=True)
    print(f"Reset controlled arrival directory: {ARRIVAL_DIRECTORY}")

dbutils.fs.mkdirs(ARRIVAL_DIRECTORY)


In [0]:
# ===================================================
# BLOCK 3 — DISCOVER SOURCE FILES (PYTHON)
# ===================================================

"""
Resolve the requested source files by their deterministic sequence numbers so
each execution represents a reproducible arrival wave.
"""

source_files = sorted(
    file_info.path
    for file_info in dbutils.fs.ls(SOURCE_DIRECTORY)
    if not file_info.isDir() and file_info.name.lower().endswith(".json")
)

assert len(source_files) == 60, (
    f"Expected 60 generated streaming files, found {len(source_files)}."
)

selected_files = source_files[BATCH_START - 1:BATCH_END]

assert len(selected_files) == BATCH_END - BATCH_START + 1

In [0]:
# ===================================================
# BLOCK 4 — STAGE CONTROLLED FILE ARRIVALS (PYTHON)
# ===================================================

"""
Copy the selected immutable source files into the watched directory. Existing
files are skipped to keep repeated notebook executions idempotent.
"""

existing_names = {
    file_info.name
    for file_info in dbutils.fs.ls(ARRIVAL_DIRECTORY)
    if not file_info.isDir()
}

staging_results = []

for source_path in selected_files:
    file_name = PurePosixPath(source_path).name
    target_path = f"{ARRIVAL_DIRECTORY}/{file_name}"

    if file_name in existing_names:
        status = "SKIPPED_EXISTING"
    else:
        copy_succeeded = dbutils.fs.cp(source_path, target_path)
        assert copy_succeeded, f"Copy failed: {source_path} -> {target_path}"
        status = "COPIED"

    staging_results.append((file_name, status, target_path))

display(
    spark.createDataFrame(
        staging_results,
        ["file_name", "status", "target_path"],
    )
)



In [0]:
# ===================================================
# BLOCK 5 — VERIFY ARRIVAL INVENTORY (PYTHON)
# ===================================================

"""
Record the current file inventory and source record count before the pipeline
update, providing an auditable reconciliation baseline.
"""

arrival_files = sorted(
    file_info.path
    for file_info in dbutils.fs.ls(ARRIVAL_DIRECTORY)
    if not file_info.isDir() and file_info.name.lower().endswith(".json")
)

arrival_df = spark.read.option("multiLine", "false").json(ARRIVAL_DIRECTORY)
arrival_record_count = arrival_df.count()

print(f"Arrival files available: {len(arrival_files):,}")
print(f"Arrival records available: {arrival_record_count:,}")
arrival_df.printSchema()
display(arrival_df.limit(10))


In [0]:
# ===================================================
# BLOCK 6 — ENFORCE EXPECTED SOURCE CONTRACT (PYTHON)
# ===================================================

"""
Fail before pipeline execution when the generated event contract differs from
the columns referenced by the streaming transformations.
"""

REQUIRED_SOURCE_COLUMNS = {
    "device_id",
    "equipment_id",
    "event_id",
    "event_timestamp_utc",
    "ingestion_timestamp_utc",
    "is_deliberately_late",
    "product_group_id",
    "site_id",
    "event_type",
    "status",
    "test_time_seconds",
}

missing_columns = REQUIRED_SOURCE_COLUMNS - set(arrival_df.columns)

assert not missing_columns, (
    f"Streaming source contract is missing columns: {sorted(missing_columns)}"
)

print("Controlled streaming arrival wave is ready for pipeline processing.")
